# Mini GPT — End-to-End Training on Colab (Task 12)

A small GPT-style transformer written from scratch in **C + CUDA + MPI**, trained
on Serbian academic text. This notebook runs the whole pipeline on a Colab T4:

1. Check the GPU
2. Clone the repo
3. Install dependencies (`colab/setup.sh`)
4. Build with **CUDA + MPI**
5. Run the full test suite (CPU + CUDA parity + MPI)
6. Data pipeline: cleaned text → BPE tokenizer → binary tokens
7. **Single-process GPU training** (watch the loss fall)
8. **Distributed training** with `mpirun -np 2`
9. Benchmark: 1 process vs 2 processes
10. Plot the loss curves

> **Before you start:** set **Runtime → Change runtime type → T4 GPU**.
> Text generation from a trained checkpoint comes in **Task 13** — this notebook
> stops at training + benchmarking.

## 1. Check the GPU

Confirms a CUDA GPU is attached. If this errors, switch to a GPU runtime.

In [ ]:
!nvidia-smi

## 2. Clone the repo

This notebook is meant to be opened straight from GitHub
(*File → Open notebook → GitHub*), so the first thing it does is clone the
project into `/content/mini-gpt` and `cd` into it. The cleaned training corpus
(`data/processed/skripta_clean.txt`) is committed, so a fresh clone already has
everything needed to train — no upload required.

In [ ]:
BRANCH = "main"   # the finished project lives on main
import os
if not os.path.isdir("/content/mini-gpt"):
    !git clone https://github.com/filiptrivan/mini-gpt.git /content/mini-gpt
%cd /content/mini-gpt
!git fetch --all --quiet && git checkout {BRANCH} && git pull --quiet
!echo "On branch:" && git rev-parse --abbrev-ref HEAD

## 3. Install dependencies

CMocka (tests), OpenMPI (distributed), PyPDF2 (optional PDF path). nvcc already ships with the GPU runtime.

In [ ]:
!bash colab/setup.sh

## 4. Build with CUDA + MPI

`ENABLE_CUDA=ON` compiles the GPU kernels and makes `train` run its
forward/backward on the GPU; `ENABLE_MPI=ON` builds the distributed pieces.
The first CUDA build takes a few minutes.

In [ ]:
!cmake -B build -DENABLE_CUDA=ON -DENABLE_MPI=ON
!cmake --build build -j2

## 5. Run the test suite

`ctest` runs everything: the CPU unit tests, the CUDA-vs-CPU parity tests
(Tasks 9–10), and the MPI distributed test (Task 11, launched under `mpirun`
with `--allow-run-as-root --oversubscribe` — Colab reports a single MPI slot, so
oversubscribe lets two ranks share it). All tests should pass.

In [ ]:
!ctest --test-dir build --output-on-failure

## 6. Data pipeline

The model needs a flat file of integer token ids. We go:

```
cleaned text  →  train_bpe  →  tokenizer (.bpe)  →  tokenize  →  tokens.bin
```

**The vocab size must match the model.** `src/train.c` is configured for
`vocab_size = 512`, and BPE produces `256 + num_merges` tokens, so we train with
**256 merges** (256 byte tokens + 256 merges = 512).

**BPE is trained on a subsample.** Our `bpe_train` uses brute-force pair counting
— `O(num_merges × length²)` (see `docs/bpe-training.md`) — so training on the
full 778 KB would take *hours*. Instead we train the tokenizer on a small,
representative slice (the merges generalize fine) and then **tokenize the full
corpus** (encoding is only `O(merges × length)` — a second or two). The model
still trains on every token of the full text.

The cleaned corpus is already in the repo. *(Optional: to use your own PDF
instead, see the commented cell at the end of this section.)*

In [ ]:
CLEAN = "data/processed/skripta_clean.txt"
!ls -la {CLEAN}
!echo "--- first lines of the corpus ---"
!head -c 400 {CLEAN}; echo

In [ ]:
# Train the BPE tokenizer on a 32 KB sample (~15 s; merges generalize), 256
# merges -> vocab_size 512 to match src/train.c. Training on the full corpus
# would take hours because pair counting is O(merges * length^2).
BPE_SAMPLE_BYTES = 32768
!head -c {BPE_SAMPLE_BYTES} {CLEAN} > data/processed/sample.txt
!./build/tools/train_bpe data/processed/sample.txt data/processed/tok.bpe 256

In [ ]:
# Encode the FULL corpus into int32 tokens the DataLoader reads directly
# (encoding is O(merges * length) — fast even on the whole text).
!./build/tools/tokenize {CLEAN} data/processed/tok.bpe data/processed/tokens.bin
!ls -la data/processed/tokens.bin

<details>
<summary><b>Optional:</b> start from your own PDF instead of the committed text</summary>

```python
from google.colab import files
up = files.upload()                       # pick a .pdf
pdf = next(iter(up))
!python3 tools/extract_pdf.py "{pdf}" data/raw/raw.txt
!./build/tools/preprocess data/raw/raw.txt data/processed/skripta_clean.txt
# then re-run the train_bpe + tokenize cells above
```
</details>

## 7. Single-process GPU training

One process, training on the GPU. The banner shows `CUDA (GPU)` and the effective
tokens/step. Watch `loss` trend **down** — that is the whole pipeline (forward →
backward → AdamW) working end to end. We `tee` the log so we can plot it later.

In [ ]:
STEPS = 500
LR = 1e-3
!./build/src/train data/processed/tokens.bin {STEPS} {LR} 2>&1 | tee train_np1.log

## 8. Distributed training with `mpirun -np 2`

Two processes ("ranks") train the same model in parallel: each reads a different
slice of the corpus, they **average their gradients** every step (MPI all-reduce),
and apply the identical update — so the two model copies stay in sync. The
effective batch per step doubles. Both ranks share the one T4.

In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 2 \
    ./build/src/train data/processed/tokens.bin {STEPS} {LR} 2>&1 | tee train_np2.log

## 9. Benchmark: 1 process vs 2 processes

Wall-clock for the same number of steps. **Expect `-np 2` to be no faster (often
slower)** here: both ranks share a single GPU and add MPI communication, so
there's no hardware to parallelize across. The point of this project is to
demonstrate the *distributed mechanism* (gradient averaging that keeps replicas
in sync) — on a real multi-GPU box, each rank would own a GPU and you'd get a
genuine speedup.

In [ ]:
import time, subprocess

def timed(cmd):
    t0 = time.time()
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    dt = time.time() - t0
    if p.returncode != 0:
        print(p.stdout[-2000:]); print(p.stderr[-2000:])
    return dt

BENCH_STEPS = 100
tok = "data/processed/tokens.bin"
t1 = timed(f"./build/src/train {tok} {BENCH_STEPS} {LR}")
t2 = timed(f"mpirun --allow-run-as-root --oversubscribe -np 2 "
           f"./build/src/train {tok} {BENCH_STEPS} {LR}")

print(f"\n{BENCH_STEPS} steps:")
print(f"  1 process : {t1:6.2f} s")
print(f"  2 processes: {t2:6.2f} s  (2x effective batch, shared GPU)")

## 10. Loss curves

Parse the logs from steps 7–8 and plot. Both should slope downward.

In [ ]:
import re
import matplotlib.pyplot as plt

def parse(path):
    steps, losses = [], []
    for line in open(path):
        m = re.match(r"step\s+(\d+)\s+\|\s+loss\s+([0-9.]+)", line)
        if m:
            steps.append(int(m.group(1)))
            losses.append(float(m.group(2)))
    return steps, losses

plt.figure(figsize=(8, 5))
for path, label in [("train_np1.log", "1 process"),
                    ("train_np2.log", "2 processes (mpirun -np 2)")]:
    try:
        s, l = parse(path)
        if s:
            plt.plot(s, l, label=label)
    except FileNotFoundError:
        pass
plt.xlabel("step"); plt.ylabel("loss"); plt.title("Mini GPT training loss")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

## Done

You trained a 534K-parameter GPT from scratch — C for the math, CUDA for the GPU
kernels, MPI for distributed gradient averaging — on Serbian academic text.

**What success looks like:** the loss curve slopes down. The model will *not*
produce coherent Serbian (534K params on a small corpus is tiny) — the value is
that the whole hand-built system trains correctly.

**Next (Task 13):** `tools/generate.c` for autoregressive sampling, so you can
feed a prompt and watch it generate text, plus the project README.